# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

La estrategia consiste en representar tanto las películas como las preferencias de cada usuario en un espacio vectorial común, y recomendar las películas cuyo vector sea más similar al perfil del usuario.

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim

   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   --------------------------------------- 588.9/588.9 kB 10.2 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import pandas as pd
import re       # libreria de expresiones regulares
import string   # libreria de cadena de caracteres
from gensim.models.phrases import Phrases, Phraser
import multiprocessing
from gensim.models import Word2Vec
from unidecode import unidecode
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [3]:
sinopsis = load_dataset("mathigatti/spanish_imdb_synopsis")

README.md: 0.00B [00:00, ?B/s]

plots.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4967 [00:00<?, ? examples/s]

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [4]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/usuarios/usuarios.csv")

In [5]:
usuarios

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...
5,U06,Martín,definido,Bienvenidos a Collinwood,El gran golpe,L.A. Confidential,Sympathy for Mr. Vengeance,La otra cara del crimen,Un grupo de personas planea un robo o estafa y...
6,U07,Sofía,definido,Velvet Goldmine,"Cuanto más, ¡mejor!",La vida de bohemia,Cero en conducta,Corazón salvaje,Una película sobre músicos o artistas que vive...
7,U08,Diego,definido,Superdetective en Hollywood,Mission: Impossible,Misión: Imposible 3,"Walker, Texas Ranger",300,Acción directa con un héroe que trabaja solo o...
8,U09,Elena,definido,Viaje a Darjeeling,Mi Idaho privado,Melinda y Melinda,La ciencia del sueño,Un beso,Algo tranquilo sobre personas que intentan rec...
9,U10,Facundo,definido,Sátántangó,Corazón salvaje,Mi Idaho privado,La ciencia del sueño,Sympathy for Mr. Vengeance,"Algo que sea difícil de clasificar, con una ló..."


Visualizamos las queries

In [6]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [7]:
df_pelis = pd.DataFrame(sinopsis['train'])
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

### Función de limpieza de texto

Aplicamos un pipeline de limpieza estándar: minúsculas, eliminación de puntuación y palabras con números. 


In [8]:
def limpiar_texto(text):
    text = unidecode(str(text)) 
    # pasa las mayusculas del texto a minusculas
    text = text.lower()
    # reemplaza texto entre corchetes por espacio en blanco
    text = re.sub(r'\[.*?¿\]%', ' ', text)
    # reemplaza signos de puntuacion por espacio en blanco
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    # remueve palabras que contienen numeros.
    text = re.sub(r'\w*\d\w*', '', text)
    # remueve caracteres especiales y saltos de linea
    text = re.sub('[‘’“”…«»]', '', text)
    text = re.sub('\n', ' ', text)
    return text

Unificamos las variables relevantes en un texto

In [9]:
df_pelis["texto"] = (
    df_pelis["name"] + " "
    + df_pelis["description"] + " "
    + df_pelis["year"].fillna('').astype(str) + " "
    + df_pelis["director"].fillna('') + " "
    # + df_pelis["genre"] + " " # revisar espanglish
    # + df_pelis["keywords"] # revisar espanglish
)

In [10]:
limpieza = lambda x: limpiar_texto(x)
data_clean = pd.DataFrame(df_pelis["texto"].apply(limpieza))

Usamos Phrases de para detectar pares de palabras que aparecen juntas con frecuencia y tienen sentido como unidad

In [11]:
input = [row.split() for row in data_clean["texto"]] # separamos en una lista
phrases = Phrases(input, min_count=10, progress_per=1000)

bigram = Phraser(phrases)

sentences = bigram[input]

## Embedding de peliculas

### Entrenamos modelo World2Vec

> [!!!] falta elegir los parametros del modelo acorde al trabajo.

In [12]:
cores = multiprocessing.cpu_count()

w2v_model = Word2Vec(min_count=10, # ignora palabras cuya frecuencia es menor a esta
                     window=8, # tamanio de la ventana de contexto
                     vector_size=300, # dimension del embedding
                     sample=6e-5, # umbral para downsamplear palabras muy frecuentes
                     alpha=0.03, # tasa de aprendizaje inicial (entrenamiento de la red neuronal)
                     min_alpha=0.0007, # tasa de aprendizaje minima
                     negative=20, # penalidad de palabras muy frecuentes o poco informaitvas
                     workers=cores) # numero de cores para entrenar el modelo

w2v_model.build_vocab(sentences, progress_per=10000) # construye el vocabulario

### ENTRENA EL MODELO
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

(1137819, 4446570)

### Calcular vector promedio de cada película

Definimos una funcion que calcula el vector promedio a partir de un texto

In [13]:
def obtener_vector_promedio(texto, modelo):
    palabras = texto.split()

    vectores_palabras = [modelo.wv[palabra] for palabra in palabras if palabra in modelo.wv]

    if not vectores_palabras:
        return np.zeros(modelo.wv.vector_size)

    return np.mean(vectores_palabras, axis=0)

Calculamos el embedding promedio de cada pelicula

In [14]:
embeddings_peliculas = pd.DataFrame(np.array([obtener_vector_promedio(text, w2v_model) for text in data_clean['texto']]))

In [15]:
embeddings_peliculas.head()

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.009753,0.095752,0.204316,0.175830,0.022951,-0.124756,0.002016,0.300940,0.004834,-0.098880,...,-0.026032,0.091463,0.330295,0.159072,0.091593,0.082100,0.112218,-0.152599,0.185008,-0.167378
1,0.021853,0.079800,0.207577,0.174201,0.028824,-0.136850,-0.016275,0.300373,0.008432,-0.068054,...,-0.020894,0.092049,0.364281,0.151342,0.063230,0.074564,0.121604,-0.145831,0.193450,-0.200411
2,0.025855,0.083569,0.199238,0.178953,0.039096,-0.147023,-0.027382,0.282505,0.000786,-0.071351,...,-0.023240,0.076178,0.400534,0.156461,0.060550,0.094877,0.136859,-0.159612,0.201700,-0.219689
3,0.021466,0.081942,0.205221,0.174446,0.031798,-0.140922,-0.020895,0.294557,0.003840,-0.070585,...,-0.023495,0.086120,0.375682,0.150316,0.066431,0.081255,0.126789,-0.150710,0.195055,-0.207570
4,0.020766,0.084809,0.202805,0.175547,0.024043,-0.138600,-0.022038,0.295414,0.012851,-0.063292,...,-0.019341,0.084846,0.357886,0.150810,0.063261,0.077032,0.118983,-0.140939,0.188698,-0.196512


Como el dataset de películas y el de embeddings se construyeron en el mismo orden,podemos unirlos directamente por índice 

In [16]:
pelis_embd = embeddings_peliculas.merge(df_pelis[["name","id"]], left_index=True, right_index=True)
pelis_embd.head()

,0,1,2,3,4,5,6,7,8,9,...,292,293,294,295,296,297,298,299,name,id
0,0.009753,0.095752,0.204316,0.175830,0.022951,-0.124756,0.002016,0.300940,0.004834,-0.098880,...,0.330295,0.159072,0.091593,0.082100,0.112218,-0.152599,0.185008,-0.167378,Herida abierta,1
1,0.021853,0.079800,0.207577,0.174201,0.028824,-0.136850,-0.016275,0.300373,0.008432,-0.068054,...,0.364281,0.151342,0.063230,0.074564,0.121604,-0.145831,0.193450,-0.200411,"Elvira, reina de las tinieblas",2
2,0.025855,0.083569,0.199238,0.178953,0.039096,-0.147023,-0.027382,0.282505,0.000786,-0.071351,...,0.400534,0.156461,0.060550,0.094877,0.136859,-0.159612,0.201700,-0.219689,Durmiendo con su enemigo,3
3,0.021466,0.081942,0.205221,0.174446,0.031798,-0.140922,-0.020895,0.294557,0.003840,-0.070585,...,0.375682,0.150316,0.066431,0.081255,0.126789,-0.150710,0.195055,-0.207570,Elizabethtown,4
4,0.020766,0.084809,0.202805,0.175547,0.024043,-0.138600,-0.022038,0.295414,0.012851,-0.063292,...,0.357886,0.150810,0.063261,0.077032,0.118983,-0.140939,0.188698,-0.196512,Godzilla,5


## Embeddings de usuarios

Aplicamos la misma función de limpieza que usamos para las sinopsis

In [17]:
data_clean_users = pd.DataFrame(usuarios["query"].apply(limpieza))

### Embedding de las queries

In [18]:
embeddings_query = pd.DataFrame()

for query in data_clean_users["query"]:
    words = query.split()
    words_embeddings = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    embedding_mean = np.mean(words_embeddings, axis=0)
    embeddings_query = pd.concat([embeddings_query, pd.DataFrame(embedding_mean).T])

embeddings_query.reset_index(drop=True,inplace=True)

In [19]:
embeddings_query = embeddings_query.merge(usuarios[["id"]], left_index=True, right_index=True)
embeddings_query

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,id
0,0.023901,0.075625,0.211497,0.177978,0.031741,-0.148525,-0.019671,0.293292,0.004687,-0.064965,...,0.073837,0.393004,0.146877,0.053748,0.082681,0.139662,-0.152274,0.200703,-0.222755,U01
1,0.016412,0.076420,0.210589,0.172183,0.026855,-0.131431,-0.009089,0.298682,0.010072,-0.062752,...,0.088716,0.350554,0.151823,0.059137,0.069474,0.127290,-0.139313,0.188390,-0.199597,U02
2,0.020254,0.077857,0.212643,0.179235,0.030669,-0.142070,-0.006671,0.300670,0.007390,-0.066111,...,0.085754,0.362313,0.151605,0.058176,0.067325,0.131892,-0.144796,0.195329,-0.205282,U03
3,0.023987,0.074179,0.206599,0.182471,0.042434,-0.156969,-0.035508,0.281570,0.005765,-0.054320,...,0.061981,0.410384,0.152965,0.047926,0.095072,0.150790,-0.151529,0.199629,-0.238138,U04
4,0.020864,0.080730,0.199831,0.173414,0.034982,-0.139705,-0.026236,0.286947,0.004040,-0.067854,...,0.084854,0.377489,0.153975,0.059507,0.086037,0.128720,-0.152642,0.191515,-0.208895,U05
5,0.015846,0.088441,0.206329,0.175643,0.028350,-0.133409,-0.009298,0.302491,0.012362,-0.079999,...,0.088015,0.344501,0.158671,0.074888,0.071518,0.114213,-0.147809,0.188807,-0.183962,U06
6,0.025156,0.065764,0.215162,0.180603,0.038550,-0.154712,-0.022262,0.290097,0.004708,-0.050128,...,0.078002,0.399757,0.153222,0.043781,0.083316,0.148288,-0.144840,0.199485,-0.235736,U07
7,0.019839,0.081724,0.205472,0.172131,0.031293,-0.139985,-0.024095,0.291336,0.004569,-0.066438,...,0.080227,0.370174,0.152782,0.062526,0.085423,0.127469,-0.147173,0.192914,-0.202640,U08
8,0.018075,0.079618,0.211832,0.181299,0.030961,-0.146536,-0.016055,0.293851,0.006831,-0.063203,...,0.075304,0.370642,0.152148,0.058740,0.077532,0.135685,-0.147409,0.193747,-0.209715,U09
9,0.022834,0.077169,0.207667,0.182041,0.037158,-0.153719,-0.026890,0.285537,0.001328,-0.062018,...,0.063966,0.397347,0.151695,0.055907,0.094163,0.144319,-0.153459,0.194254,-0.227061,U10


### Embedding historial

El historial se representa como el promedio de los embeddings de las 5 películas vistas por el usuario. 
Si alguna película del historial no se encuentra en el corpus (por diferencias de nombre), se omite del cálculo y se informa por pantalla.

In [20]:
def calcular_embedding_historial(usuario, pelis_embd):
    peliculas_usuario = usuario[['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']].tolist()
    embeddings = []
    for pelicula in peliculas_usuario:
        # check pelicula in pelis_embd
        if pelicula not in pelis_embd['name'].values:
            print(f"Película '{pelicula}' no encontrada en el DataFrame de embeddings.")
            continue # se omite del calculo del embedding del historial
        embedding = pelis_embd[pelis_embd['name'] == pelicula].iloc[0][:-2].to_numpy(dtype=np.float32)
        embeddings.append(embedding)
    historial_embedding = np.mean(embeddings, axis=0)
    return historial_embedding

In [21]:
historiales_embeddings = []
for index, usuario in usuarios.iterrows():
    historial_embedding = calcular_embedding_historial(usuario, pelis_embd)
    historial_embedding = np.append(historial_embedding, usuario['id'])
    
    historiales_embeddings.append(historial_embedding)

historiales_df = pd.DataFrame(historiales_embeddings)

Película 'Rec' no encontrada en el DataFrame de embeddings.
Película 'El secreto de sus ojos' no encontrada en el DataFrame de embeddings.
Película 'El exorcista' no encontrada en el DataFrame de embeddings.
Película 'Intocable' no encontrada en el DataFrame de embeddings.
Película 'Una mente brillante' no encontrada en el DataFrame de embeddings.
Película 'Paddington' no encontrada en el DataFrame de embeddings.


Podemos agregarlas ya que no tiene sentido que falten
> HACER

In [22]:
historiales_df

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,300
0,0.023019426,0.08107916,0.20553939,0.17817369,0.033749864,-0.14451632,-0.020602802,0.29269645,0.004206215,-0.0692743,...,0.08140372,0.3783004,0.15267166,0.06262685,0.08213409,0.13051032,-0.15079813,0.19514544,-0.21005492,U01
1,0.021005165,0.07876,0.204427,0.1775223,0.03176293,-0.14065953,-0.015804913,0.29559866,0.0046811034,-0.06773408,...,0.088093355,0.37540162,0.15615539,0.065370455,0.08114655,0.13119993,-0.14877704,0.19716962,-0.20794973,U02
2,0.02318878,0.07832129,0.20619234,0.17795745,0.03177381,-0.14261387,-0.020088967,0.29610088,0.005035936,-0.065944836,...,0.08494466,0.3747632,0.15279487,0.063219845,0.079228416,0.12972662,-0.14836818,0.19657846,-0.20987353,U03
3,0.020298257,0.08134015,0.2039313,0.17503415,0.03079218,-0.14039749,-0.019650947,0.29524708,0.0062636645,-0.07211034,...,0.08374048,0.37037086,0.15279923,0.06486895,0.08149091,0.12215135,-0.1483669,0.1936798,-0.20302837,U04
4,0.020775406,0.08107741,0.20444684,0.17659417,0.03061298,-0.13756077,-0.01830101,0.29690796,0.0074916435,-0.06779589,...,0.08847487,0.36422285,0.153651,0.06661478,0.07854045,0.1254606,-0.14682291,0.19266969,-0.199789,U05
5,0.018631807,0.086118355,0.20130856,0.17764898,0.03193862,-0.13793287,-0.017070895,0.29589573,0.005083607,-0.07460811,...,0.08415518,0.36846882,0.15894802,0.07020644,0.08653968,0.124560334,-0.14909476,0.1925672,-0.19901228,U06
6,0.018401448,0.075906835,0.20997998,0.17931035,0.028999254,-0.13759251,-0.012045505,0.29992503,0.008933194,-0.06212492,...,0.08982773,0.35820058,0.15573634,0.064661264,0.067932576,0.12684922,-0.14210574,0.19214119,-0.19939676,U07
7,0.016567264,0.085341975,0.20250487,0.17652176,0.027961921,-0.13210389,-0.014329131,0.29828173,0.0062629553,-0.07608738,...,0.09049725,0.354111,0.15774587,0.07369049,0.07967867,0.118631385,-0.14788216,0.19113898,-0.18921821,U08
8,0.021878341,0.07545775,0.2082601,0.17824915,0.032931726,-0.14366102,-0.01885568,0.29505837,0.0055368873,-0.06323238,...,0.08447723,0.37826213,0.15294616,0.057633508,0.0771456,0.13362527,-0.14780463,0.19614424,-0.21400872,U09
9,0.01999541,0.081507884,0.20482607,0.17725424,0.031715177,-0.1400293,-0.017849773,0.29637828,0.0053065643,-0.06839026,...,0.08417245,0.36914033,0.1544525,0.06332314,0.07908412,0.12757589,-0.14611621,0.19268104,-0.2043653,U10


## Recomendaciones

El perfil de cada usuario se construye como un promedio ponderado entre el embedding de su query y el embedding de su historial. Le asignamos un peso de 0.7 a la query y un 0.3 al historial con el fin de priorizar lo que el usuario quiere en ese momento.
La similitud entre el perfil del usuario y cada película se calcula con similitud coseno.

In [ ]:
movie_embeddings = pelis_embd.iloc[:, :-2].to_numpy(dtype=np.float32)

for user_id in embeddings_query['id']:
    query_embedding = embeddings_query[embeddings_query['id'] == user_id].iloc[0][:-1].to_numpy(dtype=np.float32)
    historial_embedding = historiales_df[historiales_df.iloc[:, -1] == user_id].iloc[0][:-1].to_numpy(dtype=np.float32)

    weights = [0.7, 0.3] # peso del embedding de la query y del historial respectivamente
    mean_query_historial_embedding = np.average([query_embedding, historial_embedding], axis=0, weights=weights)
    
    similarities = cosine_similarity(mean_query_historial_embedding.reshape(1, -1), movie_embeddings)
    
    top_10_indices = similarities.argsort()[0][-10:][::-1] 
    
    similar_movies_df = pd.DataFrame({
        'movie_name': pelis_embd.loc[top_10_indices, 'name'].values,
        'similarity_score': similarities[0, top_10_indices]
    })
    display(usuarios[usuarios['id'] == user_id])
    print(usuarios[usuarios['id'] == user_id]['query'].values[0])
    print(f"Top 10 Most Similar Movies for User ID {user_id}:")
    display(similar_movies_df)

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...


Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Top 10 Most Similar Movies for User ID U01:


,movie_name,similarity_score
0,La vida secreta de las palabras,0.999816
1,Whale rider,0.999805
2,Una rubia muy dudosa,0.999793
3,El grito 2,0.999781
4,Le père Noël est une ordure,0.999777
5,La angustia del miedo,0.999771
6,La camarera,0.999759
7,La chica del puente,0.999758
8,Eduardo Manostijeras,0.999751
9,La red,0.999745


,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...


Busco algo basado en hechos reales sobre corrupción o poder político
Top 10 Most Similar Movies for User ID U02:


...

Algo que pueda ver con amigos o solo, que no sea muy larga ni muy corta
Top 10 Most Similar Movies for User ID U13:


,movie_name,similarity_score
0,Zelig,0.999802
1,In-natural,0.999772
2,Arlington Road: Temerás a tu vecino,0.999762
3,Secretos de familia,0.999758
4,Novio por una noche,0.999748
5,Persiguiendo a Amy,0.999735
6,Monster House,0.999729
7,Más vale muerto,0.999726
8,Virgen a los 40,0.999712
9,Un vecino con pocas luces,0.999708


,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
13,U14,Paula,ambiguo,Lost in Translation,Transformers,Mamma Mia! La película,Réquiem por un sueño,Paddington,"No tengo ganas de pensar mucho, pero tampoco q..."


No tengo ganas de pensar mucho, pero tampoco quiero algo vacío, algo intermedio
Top 10 Most Similar Movies for User ID U14:


,movie_name,similarity_score
0,Un vecino con pocas luces,0.999724
1,Persiguiendo a Amy,0.999687
2,Big Bang,0.999682
3,Cuando menos te lo esperas... (Something&apos;...,0.999650
4,¿Qué pasa con Bob?,0.999643
5,La posesión,0.999630
6,Los Teleñecos conquistan Manhattan,0.999599
7,Llamaradas,0.999588
8,Serendipity,0.999588
9,Dos viejos gruñones,0.999587


Durante el desarrollo evaluamos distintas formas de construir el perfil del usuario para compararlo con los promedios de las peliculas y así realizar la recomendación:

- Opción 1: Promedio del query con historial por igual, no distingue entre la intención actual y las preferencias pasadas.

- Opción 2: Promedio ponderado (implementada), permite dar diferente peso a lo que el usuario quiere y a lo que el usuario ya consumio. 

- Opción 3: Promedio poderado del historial por orden de visualización. Asumiría que las películas más recientes del historial son más representativas del gusto actual. No se implementó por falta de información sobre el orden de visualización en el dataset.